# IBKR 新闻模块完整校验 notebook

这个 notebook 用来逐块检查 `news_api` 的所有模块是否正确。默认全部是离线测试，不连接 TWS / IB Gateway，不真实发送 Bark。

检查范围：配置、股票池、数据结构、清洗、事件分类、相关性、重要性评分、去重、SQLite、正文接口、Bark、服务队列、IB 回调适配、订阅管理。


## 0. 环境准备


In [1]:
from pathlib import Path
import json
import sys
import tempfile

cwd = Path.cwd()
package_dir = cwd if cwd.name == "news_api" else cwd / "news_api"
sys.path.insert(0, str(package_dir.parent))

print("package_dir =", package_dir)
assert package_dir.exists(), "没有找到 news_api 目录，请确认当前 notebook 位置。"

results = {}

def run_case(name, func):
    result = func()
    results[name] = result
    print(f"[OK] {name}")
    return result


package_dir = e:\策略\IB-API\news_api


## 1. 包导入检查


In [2]:
def test_imports():
    import news_api
    from news_api import ArticleContent, NewsAnalysis, NewsHeadline, NewsService, SQLiteNewsStorage
    from news_api import article_fetcher, bark_client, cleaner, config, deduplicator
    from news_api import event_classifier, ib_client, importance_scorer, relevance
    from news_api import service, storage, subscription_manager, watchlist
    assert hasattr(news_api, "NewsService")
    return {"package": news_api.__name__}

run_case("imports", test_imports)


[OK] imports


{'package': 'news_api'}

## 2. config.py 配置检查


In [5]:
def test_config():
    from news_api.config import DEFAULT_PROVIDER_CODES, NewsSettings, SETTINGS
    assert "DJ-N" in DEFAULT_PROVIDER_CODES
    assert SETTINGS.article_fetch_score < SETTINGS.default_push_score
    assert SETTINGS.portfolio_push_score <= SETTINGS.default_push_score
    custom = NewsSettings(port=4002, client_id=123, db_path=package_dir / "data" / "test.sqlite")
    assert custom.port == 4002
    assert custom.client_id == 123
    assert custom.db_path.name == "test.sqlite"
    return custom

run_case("config.py", test_config)


[OK] config.py


NewsSettings(host='127.0.0.1', port=4002, client_id=123, provider_codes='BRFG+BRFUPDN+DJ-N+DJ-RTA+DJ-RTE+DJ-RTG+DJ-RTPRO+DJNL', local_timezone='Asia/Shanghai', db_path=WindowsPath('e:/策略/IB-API/news_api/data/test.sqlite'), history_buffer_minutes=5, max_historical_results=300, article_fetch_score=40, default_push_score=70, portfolio_push_score=60, bark_key='', bark_base_url='https://api.day.app', dashboard_url='')

## 3. watchlist.py 股票池检查


In [6]:
def test_watchlist():
    from news_api.watchlist import WATCHLIST, normalize_watchlist, portfolio_symbols
    custom = {
        "orcl": {"exchange": "NYSE", "priority": 0, "aliases": ["Oracle", "OCI"]},
        "avav": {"exchange": "NASDAQ", "priority": 1, "aliases": ["AeroVironment"]},
    }
    normalized = normalize_watchlist(custom)
    assert "ORCL" in normalized
    assert normalized["ORCL"]["aliases"][0] == "ORCL"
    assert normalized["ORCL"]["priority"] == 0
    assert portfolio_symbols(normalized) == {"ORCL"}
    assert "SMCI" in normalize_watchlist(WATCHLIST)
    return normalized

run_case("watchlist.py", test_watchlist)


[OK] watchlist.py


{'ORCL': {'exchange': 'NYSE',
  'priority': 0,
  'aliases': ['ORCL', 'Oracle', 'OCI']},
 'AVAV': {'exchange': 'NASDAQ',
  'priority': 1,
  'aliases': ['AVAV', 'AeroVironment']}}

## 4. models.py 数据结构检查


In [7]:
def test_models():
    from news_api.models import ArticleContent, NewsAnalysis, NewsHeadline, utc_now_iso
    event = NewsHeadline("ORCL", "DJ-N", "001", "Oracle raises guidance", "2026-06-15T13:30:00Z")
    article = ArticleContent(provider="DJ-N", article_id="001", article_text="Oracle text", fetch_status="ok")
    analysis = NewsAnalysis(symbol="ORCL", provider="DJ-N", article_id="001", headline=event.headline)
    assert event.unique_key == "DJ-N:001"
    assert article.fetch_status == "ok"
    assert analysis.event_type == "OTHER"
    assert "T" in utc_now_iso()
    return {"event": event, "article": article, "analysis": analysis}

run_case("models.py", test_models)


[OK] models.py


{'event': NewsHeadline(symbol='ORCL', provider='DJ-N', article_id='001', headline='Oracle raises guidance', published_at='2026-06-15T13:30:00Z', received_at='2026-06-16T04:48:41.223990+00:00', ticker_id=None, con_id=None, headline_raw='', publisher='', extra_data=''),
 'article': ArticleContent(provider='DJ-N', article_id='001', article_text='Oracle text', article_html='', publisher='', fetch_status='ok', fetched_at='2026-06-16T04:48:41.223990+00:00', error=''),
 'analysis': NewsAnalysis(symbol='ORCL', provider='DJ-N', article_id='001', headline='Oracle raises guidance', event_type='OTHER', event_subtypes=[], relevance_score=0, event_score=0, novelty_score=10, source_score=5, market_impact_score=0, importance_score=0, sentiment=0.0, summary_zh=[], reason_important='', story_id='', should_fetch_article=False, should_push=False)}

## 5. cleaner.py 清洗逻辑检查


In [8]:
def test_cleaner():
    from news_api.cleaner import clean_article_text, clean_headline, normalize_text, parse_headline_metadata, parse_ib_time
    raw = "{A:800015:L:en}Oracle raises guidance after quarterly results -- Dow Jones"
    parsed = parse_headline_metadata(raw)
    assert parsed["headline"] == "Oracle raises guidance after quarterly results"
    assert parsed["publisher"] == "Dow Jones"
    assert parsed["language"] == "en"
    assert clean_headline(raw) == parsed["headline"]
    utc_time, local_time = parse_ib_time("2026-06-15 13:30:00.0", "Asia/Shanghai")
    assert utc_time.startswith("2026-06-15T13:30:00")
    assert local_time.startswith("2026-06-15T21:30:00")
    assert normalize_text("Oracle, Inc. raises guidance!") == "oracle inc raises guidance"
    text = clean_article_text("<html><script>x</script><p>Line 1</p><p>Line 2</p>Copyright (c) test</html>")
    assert "Line 1" in text and "Line 2" in text
    assert "script" not in text.lower()
    assert "Copyright" not in text
    return {"parsed": parsed, "article_text": text}

run_case("cleaner.py", test_cleaner)


[OK] cleaner.py


{'parsed': {'headline': 'Oracle raises guidance after quarterly results',
  'publisher': 'Dow Jones',
  'language': 'en',
  'metadata': {'A': '800015', 'L': 'en'}},
 'article_text': 'Line 1\n\nLine 2'}

## 6. event_classifier.py 事件分类检查


In [9]:
def test_event_classifier():
    from news_api.event_classifier import classify_headline
    cases = [
        ("Oracle raises guidance after earnings", "Dow Jones", "EARNINGS"),
        ("Company wins government contract", "Briefing", "CONTRACT"),
        ("CEO resigns after board review", "Reuters", "MANAGEMENT"),
        ("Shareholder alert reminds investors of class action deadline", "Law Firm", "LOW_VALUE"),
        ("A normal market comment", "Blog", "OTHER"),
    ]
    output = []
    for headline, publisher, expected in cases:
        event_type, hits, score = classify_headline(headline, publisher)
        assert event_type == expected, (headline, event_type, expected)
        assert score >= 0
        output.append((headline, event_type, hits, score))
    return output

run_case("event_classifier.py", test_event_classifier)


[OK] event_classifier.py


[('Oracle raises guidance after earnings', 'EARNINGS', ['earnings'], 26),
 ('Company wins government contract', 'CONTRACT', ['government contract'], 22),
 ('CEO resigns after board review', 'MANAGEMENT', ['ceo resigns'], 20),
 ('Shareholder alert reminds investors of class action deadline',
  'LOW_VALUE',
  ['shareholder alert', 'class action deadline', 'reminds investors'],
  0),
 ('A normal market comment', 'OTHER', [], 4)]

## 7. relevance.py 相关性评分检查


In [10]:
def test_relevance():
    from news_api.relevance import score_relevance
    aliases = ["ORCL", "Oracle", "Oracle Corp", "OCI"]
    high = score_relevance("ORCL", aliases, "Oracle raises guidance for OCI cloud business", "Oracle Oracle Oracle Oracle Oracle expands cloud capacity")
    medium = score_relevance("ORCL", aliases, "Cloud software company reports results", "Oracle mentioned once")
    low = score_relevance("ORCL", aliases, "Apple launches new product", "No related company here")
    assert high == 30
    assert medium >= low
    assert low == 0
    return {"high": high, "medium": medium, "low": low}

run_case("relevance.py", test_relevance)


[OK] relevance.py


{'high': 30, 'medium': 2, 'low': 0}

## 8. importance_scorer.py 重要性评分检查


In [ ]:
def test_importance_scorer():
    from news_api.importance_scorer import score_news
    from news_api.models import NewsHeadline
    from news_api.watchlist import normalize_watchlist
    aliases = normalize_watchlist()["ORCL"]["aliases"]
    high_event = NewsHeadline("ORCL", "DJ-N", "score-001", "Oracle raises guidance after quarterly results", "2026-06-15T13:30:00Z", publisher="Dow Jones")
    high = score_news(high_event, aliases)
    assert high.event_type in {"EARNINGS", "GUIDANCE"}
    assert high.importance_score >= 70
    assert high.should_fetch_article is True
    assert high.should_push is True
    low_event = NewsHeadline("ORCL", "WEB", "score-002", "Weekly review of stock market today", "2026-06-15T13:30:00Z", publisher="Unknown")
    low = score_news(low_event, aliases)
    assert low.event_type == "LOW_VALUE"
    assert low.importance_score < high.importance_score
    assert low.should_push is False
    return {"high": high, "low": low}

run_case("importance_scorer.py", test_importance_scorer)


## 9. deduplicator.py 故事去重检查


In [ ]:
def test_deduplicator():
    from news_api.deduplicator import StoryDeduplicator, headline_similarity
    from news_api.models import NewsHeadline
    ratio = headline_similarity("Dow Jones Futures Rise, Oil Skids", "Dow Jones Futures Rise, Oil Dives")
    assert ratio > 0.75
    dedup = StoryDeduplicator(threshold=0.80)
    first = NewsHeadline("ORCL", "DJ-N", "dedup-001", "Oracle raises guidance after earnings", "2026-06-15")
    second = NewsHeadline("ORCL", "DJ-N", "dedup-002", "Oracle raises guidance after quarterly earnings", "2026-06-15")
    third = NewsHeadline("SMCI", "DJ-N", "dedup-003", "Oracle raises guidance after quarterly earnings", "2026-06-15")
    story_1, is_new_1 = dedup.assign_story_id(first)
    story_2, is_new_2 = dedup.assign_story_id(second)
    story_3, is_new_3 = dedup.assign_story_id(third)
    assert is_new_1 is True
    assert is_new_2 is False
    assert story_1 == story_2
    assert is_new_3 is True
    assert story_3 != story_1
    return {"similarity": ratio, "story_1": story_1, "story_3": story_3}

run_case("deduplicator.py", test_deduplicator)


## 10. storage.py SQLite 表和读写检查


In [ ]:
def test_storage():
    from news_api.models import ArticleContent, NewsHeadline
    from news_api.importance_scorer import score_news
    from news_api.storage import SQLiteNewsStorage
    tmp_dir = Path(tempfile.mkdtemp())
    storage = SQLiteNewsStorage(tmp_dir / "news.sqlite")
    event = NewsHeadline("ORCL", "DJ-N", "store-001", "Oracle raises guidance after earnings", "2026-06-15T13:30:00Z", publisher="Dow Jones")
    assert storage.save_raw(event) is True
    assert storage.save_raw(event) is False
    article = ArticleContent(provider="DJ-N", article_id="store-001", article_text="Oracle raises guidance because cloud demand improves.", fetch_status="ok")
    storage.save_article(article)
    analysis = score_news(event, ["ORCL", "Oracle"])
    analysis.story_id = "ORCL_DJ-N_demo"
    event_id = storage.save_event(analysis)
    assert event_id >= 1
    storage.save_push_log(event.unique_key, "bark", "skipped", "no key")
    storage.set_state("last_seen:ORCL", event.published_at)
    assert storage.get_state("last_seen:ORCL") == event.published_at
    rows = storage.fetch_recent_events(limit=5)
    assert len(rows) == 1
    assert rows[0]["unique_key"] == event.unique_key
    with storage.connect() as conn:
        raw_count = conn.execute("SELECT COUNT(*) AS n FROM news_raw").fetchone()["n"]
        article_count = conn.execute("SELECT COUNT(*) AS n FROM news_articles").fetchone()["n"]
        push_count = conn.execute("SELECT COUNT(*) AS n FROM news_push_log").fetchone()["n"]
    assert raw_count == 1
    assert article_count == 1
    assert push_count == 1
    return {"db_path": storage.db_path, "event": rows[0]}

run_case("storage.py", test_storage)


## 11. article_fetcher.py 正文接口检查


In [ ]:
def test_article_fetcher():
    from news_api.article_fetcher import NoopArticleFetcher
    from news_api.models import NewsHeadline
    fetcher = NoopArticleFetcher()
    event = NewsHeadline("ORCL", "DJ-N", "fetch-001", "Oracle headline", "2026-06-15", publisher="Dow Jones")
    article = fetcher.fetch(event)
    assert article.provider == event.provider
    assert article.article_id == event.article_id
    assert article.publisher == "Dow Jones"
    assert article.fetch_status == "skipped"
    assert article.article_text == ""
    return article

run_case("article_fetcher.py", test_article_fetcher)


## 12. bark_client.py Bark 跳过逻辑检查


In [ ]:
def test_bark_client():
    from news_api.bark_client import BarkClient
    from news_api.models import NewsAnalysis
    analysis = NewsAnalysis(symbol="ORCL", provider="DJ-N", article_id="bark-001", headline="Oracle raises guidance", event_type="GUIDANCE", importance_score=86, sentiment=0.4, summary_zh=["Oracle 上调指引", "云业务需求改善"])
    client = BarkClient(key="")
    status, response = client.push(analysis, priority=0)
    assert status == "skipped"
    assert "BARK_KEY" in response
    return {"status": status, "response": response}

run_case("bark_client.py", test_bark_client)


## 13. service.py 完整离线流水线检查


In [ ]:
def test_service_pipeline():
    from news_api.article_fetcher import NoopArticleFetcher
    from news_api.bark_client import BarkClient
    from news_api.config import NewsSettings
    from news_api.models import NewsHeadline
    from news_api.service import NewsService
    from news_api.storage import SQLiteNewsStorage
    tmp_dir = Path(tempfile.mkdtemp())
    settings = NewsSettings(db_path=tmp_dir / "service.sqlite")
    storage = SQLiteNewsStorage(settings.db_path)
    service = NewsService(settings=settings, storage=storage, article_fetcher=NoopArticleFetcher(), bark_client=BarkClient(key=""))
    event = NewsHeadline("ORCL", "DJ-N", "svc-001", "Oracle raises guidance after quarterly results", "2026-06-15T13:30:00Z", publisher="Dow Jones")
    service.start()
    assert service.ingest_headline(event) is True
    service.queue.join()
    service.stop()
    assert service.ingest_headline(event) is False
    events = storage.fetch_recent_events()
    assert len(events) == 1
    assert events[0]["symbol"] == "ORCL"
    assert events[0]["importance_score"] >= 70
    assert events[0]["should_push"] == 1
    assert storage.get_state("last_seen:ORCL") == event.published_at
    with storage.connect() as conn:
        push_count = conn.execute("SELECT COUNT(*) AS n FROM news_push_log").fetchone()["n"]
        article_count = conn.execute("SELECT COUNT(*) AS n FROM news_articles").fetchone()["n"]
    assert push_count == 1
    assert article_count == 1
    return {"db_path": settings.db_path, "event": events[0], "push_count": push_count}

run_case("service.py", test_service_pipeline)


## 14. service.ingest_tick_news 回调入口检查


In [ ]:
def test_ingest_tick_news():
    from news_api.article_fetcher import NoopArticleFetcher
    from news_api.bark_client import BarkClient
    from news_api.config import NewsSettings
    from news_api.service import NewsService
    from news_api.storage import SQLiteNewsStorage
    tmp_dir = Path(tempfile.mkdtemp())
    settings = NewsSettings(db_path=tmp_dir / "tick.sqlite")
    storage = SQLiteNewsStorage(settings.db_path)
    service = NewsService(settings=settings, storage=storage, article_fetcher=NoopArticleFetcher(), bark_client=BarkClient(key=""))
    service.start()
    inserted = service.ingest_tick_news(symbol="ORCL", timestamp="2026-06-15T13:30:00Z", provider="DJ-N", article_id="tick-001", headline="{A:1:L:en}Oracle raises guidance after earnings -- Dow Jones", ticker_id=7000, extra_data="{}")
    service.queue.join()
    service.stop()
    assert inserted is True
    rows = storage.fetch_recent_events()
    assert len(rows) == 1
    assert rows[0]["headline"] == "Oracle raises guidance after earnings"
    with storage.connect() as conn:
        raw = conn.execute("SELECT * FROM news_raw WHERE unique_key='DJ-N:tick-001'").fetchone()
    assert raw["publisher"] == "Dow Jones"
    assert raw["ticker_id"] == 7000
    return {"raw": dict(raw), "event": rows[0]}

run_case("service.ingest_tick_news", test_ingest_tick_news)


## 15. subscription_manager.py 订阅管理离线检查


In [ ]:
def test_subscription_manager_offline():
    import news_api.subscription_manager as sm
    from news_api.subscription_manager import SubscriptionManager
    class FakeContract:
        def __init__(self, symbol, primary_exchange):
            self.symbol = symbol
            self.primaryExchange = primary_exchange
    class FakeClient:
        def __init__(self):
            self.ticker_id_to_symbol = {}
            self.calls = []
        def reqMktData(self, ticker_id, contract, generic_ticks, snapshot, regulatory_snapshot, options):
            self.calls.append({"ticker_id": ticker_id, "symbol": contract.symbol, "primaryExchange": contract.primaryExchange, "generic_ticks": generic_ticks, "snapshot": snapshot, "regulatory_snapshot": regulatory_snapshot, "options": options})
    original_stock_contract = sm.stock_contract
    sm.stock_contract = lambda symbol, exchange: FakeContract(symbol, exchange)
    try:
        client = FakeClient()
        manager = SubscriptionManager(client, start_ticker_id=9000)
        watchlist = {"ORCL": {"exchange": "NYSE", "priority": 0}, "AVAV": {"exchange": "NASDAQ", "priority": 1}, "P2XX": {"exchange": "NYSE", "priority": 2}}
        result = manager.subscribe_watchlist(watchlist, "DJ-N+BRFG")
    finally:
        sm.stock_contract = original_stock_contract
    assert result == {"ORCL": 9000, "AVAV": 9001}
    assert client.ticker_id_to_symbol == {9000: "ORCL", 9001: "AVAV"}
    assert len(client.calls) == 2
    assert client.calls[0]["generic_ticks"] == "mdoff,292:DJ-N+BRFG"
    assert all(call["snapshot"] is False for call in client.calls)
    return {"result": result, "calls": client.calls}

run_case("subscription_manager.py", test_subscription_manager_offline)


## 16. ib_client.py tickNews 适配离线检查


In [ ]:
def test_ib_client_ticknews_offline():
    from news_api.ib_client import IBNewsClient
    class FakeService:
        def __init__(self):
            self.events = []
        def ingest_tick_news(self, **kwargs):
            self.events.append(kwargs)
            return True
    service = FakeService()
    client = IBNewsClient.__new__(IBNewsClient)
    client.service = service
    client.ticker_id_to_symbol = {7000: "ORCL"}
    client.tickNews(tickerId=7000, timeStamp=1781530200, providerCode="DJ-N", articleId="ib-001", headline="Oracle raises guidance", extraData="{}")
    client.tickNews(tickerId=9999, timeStamp=1781530200, providerCode="DJ-N", articleId="ib-002", headline="Unknown ticker", extraData="{}")
    assert len(service.events) == 1
    assert service.events[0]["symbol"] == "ORCL"
    assert service.events[0]["provider"] == "DJ-N"
    assert service.events[0]["article_id"] == "ib-001"
    return service.events[0]

run_case("ib_client.py tickNews", test_ib_client_ticknews_offline)


## 17. 可选：真实 IB 连接和订阅检查

下面这个单元默认不连接 IB。你确认 TWS / IB Gateway 已打开、端口和新闻权限都没问题后，再把 `RUN_LIVE_IB_TEST` 改成 `True`。


In [11]:
RUN_LIVE_IB_TEST = True

if RUN_LIVE_IB_TEST:
    from news_api.config import SETTINGS
    from news_api.ib_client import IBNewsClient
    from news_api.service import NewsService
    from news_api.subscription_manager import SubscriptionManager
    from news_api.watchlist import normalize_watchlist
    service = NewsService(settings=SETTINGS)
    client = IBNewsClient(service)
    client.start_api(SETTINGS.host, SETTINGS.port, SETTINGS.client_id)
    watchlist = normalize_watchlist()
    manager = SubscriptionManager(client)
    subscribed = manager.subscribe_watchlist(watchlist, SETTINGS.provider_codes)
    print(subscribed)
else:
    print("跳过真实 IB 连接测试。需要时把 RUN_LIVE_IB_TEST 改成 True。")


{'ORCL': 7000, 'SMCI': 7001, 'AVAV': 7002}


## 18. 汇总


In [ ]:
print("已完成测试模块：")
for key in results:
    print("-", key)
assert len(results) >= 15
print("\n全部离线模块测试通过。")
